In [1]:
grid = [
    [".", "M", ".", "G"],
    [".", "#", "I", "H"],
    ["S", ".", ".", "."]
]
for row in grid:
    print(row)

['.', 'M', '.', 'G']
['.', '#', 'I', 'H']
['S', '.', '.', '.']


In [2]:
rewards = {
    "S": -1,
    ".": -1,
    "M": -3,
    "I": -1,
    "G": 10,
    "H": -10
}
print(rewards)

{'S': -1, '.': -1, 'M': -3, 'I': -1, 'G': 10, 'H': -10}


In [3]:
actions = {
    "U": (-1, 0),  # Up
    "R": (0, 1),   # Right
    "D": (1, 0),   # Down
    "L": (0, -1)   # Left
}
print(actions)

{'U': (-1, 0), 'R': (0, 1), 'D': (1, 0), 'L': (0, -1)}


In [4]:
states = []

for r in range(len(grid)):
    for c in range(len(grid[0])):
        if grid[r][c] != "#":
            states.append((r, c))
print(states)
print("Number of states:", len(states))

[(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3)]
Number of states: 11


In [5]:
def move(state, action):
    r, c = state

    dr, dc = actions[action]

    new_r = r + dr
    new_c = c + dc

    # outside map
    if new_r < 0 or new_r >= len(grid):
        return state

    if new_c < 0 or new_c >= len(grid[0]):
        return state

    # wall
    if grid[new_r][new_c] == "#":
        return state

    return (new_r, new_c)
print(move((0, 0), "L"))
print(move((0, 0), "U"))
print(move((1, 0), "R"))
print(move((2, 0), "R"))

(0, 0)
(0, 0)
(1, 0)
(2, 1)


In [6]:
V = {}

for state in states:
    V[state] = 0

print(V)

{(0, 0): 0, (0, 1): 0, (0, 2): 0, (0, 3): 0, (1, 0): 0, (1, 2): 0, (1, 3): 0, (2, 0): 0, (2, 1): 0, (2, 2): 0, (2, 3): 0}


In [7]:
terminal_states = [
    (0, 3),  # G
    (1, 3)   # H
]
print(terminal_states)

[(0, 3), (1, 3)]


In [59]:
def get_reward(state):
    r, c = state
    tile = grid[r][c]
    return rewards[tile]

In [11]:
V1 = {}

for state in states:

    if state in terminal_states:
        V1[state] = 0
        continue

    best_value = float("-inf")

    for action in actions:
        next_state = move(state, action)

        reward = get_reward(next_state)

        value = reward + gamma * V[next_state]

        best_value = max(best_value, value)

    V1[state] = best_value

print(V1)

{(0, 0): -1.0, (0, 1): -1.0, (0, 2): 10.0, (0, 3): 0, (1, 0): -1.0, (1, 2): -1.0, (1, 3): 0, (2, 0): -1.0, (2, 1): -1.0, (2, 2): -1.0, (2, 3): -1.0}


In [12]:
for r in range(len(grid)):
    row = []

    for c in range(len(grid[0])):

        if grid[r][c] == "#":
            row.append("#####")

        else:
            row.append(f"{V1[(r,c)]:5.1f}")

    print(" ".join(row))

 -1.0  -1.0  10.0   0.0
 -1.0 #####  -1.0   0.0
 -1.0  -1.0  -1.0  -1.0


In [13]:
V = V1.copy()

print(V[(0,1)])
print(V[(0,2)])
print(V[(1,2)])

-1.0
10.0
-1.0


In [14]:
state = (0, 1)

for action in actions:
    next_state = move(state, action)

    reward = get_reward(next_state)

    value = reward + gamma * V[next_state]

    print(action, next_state, value)

U (0, 1) -3.9
R (0, 2) 8.0
D (0, 1) -3.9
L (0, 0) -1.9


In [15]:
state = (0, 2)

for action in actions:
    next_state = move(state, action)

    reward = get_reward(next_state)

    value = reward + gamma * V[next_state]

    print(action, next_state, value)

U (0, 2) 8.0
R (0, 3) 10.0
D (1, 2) -1.9
L (0, 1) -3.9


In [21]:
state = (1, 2)

results = {}

# Up
results["Up"] = (
    0.8 * (get_reward(move(state, "U")) + gamma * V[move(state, "U")])
    + 0.1 * (get_reward(move(state, "L")) + gamma * V[move(state, "L")])
    + 0.1 * (get_reward(move(state, "R")) + gamma * V[move(state, "R")])
)

# Right
results["Right"] = (
    0.8 * (get_reward(move(state, "R")) + gamma * V[move(state, "R")])
    + 0.1 * (get_reward(move(state, "U")) + gamma * V[move(state, "U")])
    + 0.1 * (get_reward(move(state, "D")) + gamma * V[move(state, "D")])
)

# Down
results["Down"] = (
    0.8 * (get_reward(move(state, "D")) + gamma * V[move(state, "D")])
    + 0.1 * (get_reward(move(state, "L")) + gamma * V[move(state, "L")])
    + 0.1 * (get_reward(move(state, "R")) + gamma * V[move(state, "R")])
)

# Left
results["Left"] = (
    0.8 * (get_reward(move(state, "L")) + gamma * V[move(state, "L")])
    + 0.1 * (get_reward(move(state, "U")) + gamma * V[move(state, "U")])
    + 0.1 * (get_reward(move(state, "D")) + gamma * V[move(state, "D")])
)

for action, value in results.items():
    print(action, "=", round(value, 2))


Up = 5.21
Right = -7.39
Down = -2.71
Left = -0.91


In [24]:
def is_ice(state):
    r, c = state
    return grid[r][c] == "I"
print(is_ice((1,2)))
print(is_ice((0,0)))

True
False


In [27]:
def action_value(state, action, V):

    # ordinary tile
    if not is_ice(state):
        next_state = move(state, action)

        reward = get_reward(next_state)

        return reward + gamma * V[next_state]

    # ice tile
    if action == "U":
        directions = [("U", 0.8), ("L", 0.1), ("R", 0.1)]

    elif action == "R":
        directions = [("R", 0.8), ("U", 0.1), ("D", 0.1)]

    elif action == "D":
        directions = [("D", 0.8), ("L", 0.1), ("R", 0.1)]

    else:  # L
        directions = [("L", 0.8), ("U", 0.1), ("D", 0.1)]

    total = 0

    for direction, probability in directions:
        next_state = move(state, direction)

        reward = get_reward(next_state)

        total += probability * (
            reward + gamma * V[next_state]
        )

    return total
print(round(action_value((1,2), "U", V), 2))
print(round(action_value((1,2), "R", V), 2))
print(round(action_value((1,2), "D", V), 2))
print(round(action_value((1,2), "L", V), 2))

5.21
-7.39
-2.71
-0.91


In [28]:
V2 = {}

for state in states:

    if state in terminal_states:
        V2[state] = 0
        continue

    values = []

    for action in actions:
        values.append(
            action_value(state, action, V)
        )

    V2[state] = max(values)

print(V2)

{(0, 0): -1.9, (0, 1): 8.0, (0, 2): 10.0, (0, 3): 0, (1, 0): -1.9, (1, 2): 5.21, (1, 3): 0, (2, 0): -1.9, (2, 1): -1.9, (2, 2): -1.9, (2, 3): -1.9}


In [29]:
V = {state: 0 for state in states}

iterations = 0
delta = float("inf")

while delta > 0.000001 and iterations < 10000:

    new_V = {}

    delta = 0

    for state in states:

        if state in terminal_states:
            new_V[state] = 0
            continue

        values = []

        for action in actions:
            values.append(
                action_value(state, action, V)
            )

        new_V[state] = max(values)

        delta = max(
            delta,
            abs(new_V[state] - V[state])
        )

    V = new_V

    iterations += 1

print("Iterations:", iterations)
print("Final change:", delta)

Iterations: 12
Final change: 2.1652931048166124e-07


In [30]:
for r in range(len(grid)):
    row = []

    for c in range(len(grid[0])):

        if grid[r][c] == "#":
            row.append("#####")

        else:
            row.append(f"{V[(r,c)]:7.2f}")

    print(" ".join(row))

   4.20    8.00   10.00    0.00
   2.78 #####    5.82    0.00
   1.54    2.82    4.24    2.82


In [32]:
arrows = {
    "U": "↑",
    "R": "→",
    "D": "↓",
    "L": "←"
}
print(arrows)

{'U': '↑', 'R': '→', 'D': '↓', 'L': '←'}


In [33]:
policy = {}

for state in states:

    if state in terminal_states:
        continue

    best_action = None
    best_value = float("-inf")

    for action in actions:

        value = action_value(state, action, V)

        if value > best_value:
            best_value = value
            best_action = action

    policy[state] = best_action

print(policy)

{(0, 0): 'R', (0, 1): 'R', (0, 2): 'R', (1, 0): 'U', (1, 2): 'U', (2, 0): 'R', (2, 1): 'R', (2, 2): 'U', (2, 3): 'L'}


In [34]:
for r in range(len(grid)):
    row = []

    for c in range(len(grid[0])):

        if grid[r][c] == "#":
            row.append("#")

        elif (r, c) == (0, 3):
            row.append("G")

        elif (r, c) == (1, 3):
            row.append("H")

        else:
            row.append(arrows[policy[(r, c)]])

    print(" ".join(row))

→ → → G
↑ # ↑ H
→ → ↑ ←


In [35]:
extended_grid = [
    [".", "M", ".", "G"],
    [".", "#", "I", "H"],
    ["S", "D", ".", "W"]
]

for row in extended_grid:
    print(row)

['.', 'M', '.', 'G']
['.', '#', 'I', 'H']
['S', 'D', '.', 'W']


In [36]:
extended_rewards = {
    "S": -1,
    ".": -1,
    "M": -3,
    "I": -1,
    "D": -2,
    "W": -5,
    "G": 10,
    "H": -10
}

print(extended_rewards)

{'S': -1, '.': -1, 'M': -3, 'I': -1, 'D': -2, 'W': -5, 'G': 10, 'H': -10}


In [37]:
grid = extended_grid
rewards = extended_rewards

print(grid)

[['.', 'M', '.', 'G'], ['.', '#', 'I', 'H'], ['S', 'D', '.', 'W']]


In [38]:
states = []

for r in range(len(grid)):
    for c in range(len(grid[0])):
        if grid[r][c] != "#":
            states.append((r, c))

print(states)
print("Number of states:", len(states))

[(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3)]
Number of states: 11


In [39]:
V = {state: 0 for state in states}

iterations = 0
delta = float("inf")

while delta > 0.000001 and iterations < 10000:

    new_V = {}
    delta = 0

    for state in states:

        if state in terminal_states:
            new_V[state] = 0
            continue

        values = []

        for action in actions:
            values.append(
                action_value(state, action, V)
            )

        new_V[state] = max(values)

        delta = max(
            delta,
            abs(new_V[state] - V[state])
        )

    V = new_V
    iterations += 1

print("Iterations:", iterations)
print("Final change:", delta)

Iterations: 11
Final change: 2.405881232014906e-07


In [40]:
for r in range(len(grid)):
    row = []

    for c in range(len(grid[0])):

        if grid[r][c] == "#":
            row.append("#####")

        else:
            row.append(f"{V[(r,c)]:7.2f}")

    print(" ".join(row))

   4.20    8.00   10.00    0.00
   2.78 #####    5.82    0.00
   1.50    2.82    4.24    2.82


In [41]:
policy = {}

for state in states:

    if state in terminal_states:
        continue

    best_action = None
    best_value = float("-inf")

    for action in actions:

        value = action_value(state, action, V)

        if value > best_value:
            best_value = value
            best_action = action

    policy[state] = best_action

In [42]:
for r in range(len(grid)):
    row = []

    for c in range(len(grid[0])):

        if grid[r][c] == "#":
            row.append("#")

        elif (r, c) == (0, 3):
            row.append("G")

        elif (r, c) == (1, 3):
            row.append("H")

        else:
            row.append(arrows[policy[(r, c)]])

    print(" ".join(row))

→ → → G
↑ # ↑ H
↑ → ↑ ←


In [43]:
print(move((0,0), "U"))
print(move((0,0), "L"))
print(move((1,0), "R"))


(0, 0)
(0, 0)
(1, 0)


In [44]:
print(0.8 + 0.1 + 0.1)

1.0


In [45]:
print("Mud =", rewards["M"])
print("Sand =", rewards["D"])
print("Water =", rewards["W"])

Mud = -3
Sand = -2
Water = -5


In [61]:
print("Goal terminal:", (0, 3) in terminal_states)
print("Hole terminal:", (1, 3) in terminal_states)

print("No rewards are collected after reaching Goal or Hole because the episode terminates immediately.")

Goal terminal: True
Hole terminal: True
No rewards are collected after reaching Goal or Hole because the episode terminates immediately.


In [46]:
import random
start_state = (2, 0)
print(start_state)

(2, 0)


In [52]:
def simulate_move(state, action):

    if not is_ice(state):
        return move(state, action)

    if action == "U":
        directions = ["U", "L", "R"]

    elif action == "R":
        directions = ["R", "U", "D"]

    elif action == "D":
        directions = ["D", "L", "R"]

    else:
        directions = ["L", "U", "D"]

    chosen = random.choices(
        directions,
        weights=[0.8, 0.1, 0.1]
    )[0]

    return move(state, chosen)

In [53]:
for i in range(10):
    print(simulate_move((1,2), "U"))

(0, 2)
(0, 2)
(1, 3)
(0, 2)
(0, 2)
(0, 2)
(0, 2)
(0, 2)
(0, 2)
(0, 2)


In [57]:
def run_episode():

    state = start_state
    steps = 0
    discounted_return = 0
    discount = 1

    while steps < 100:

        if state == (0, 3):
            return "goal", steps, discounted_return

        if state == (1, 3):
            return "hole", steps, discounted_return

        action = policy[state]

        next_state = simulate_move(state, action)

        reward = get_reward(next_state)

        discounted_return += discount * reward

        discount *= 0.9

        state = next_state

        steps += 1

    return "time_limit", steps, discounted_return
print(run_episode())

('goal', 5, 1.5020000000000016)


In [65]:
import random

start_state = (2, 0)

def simulate_move(state, action):

    if not is_ice(state):
        return move(state, action)

    if action == "U":
        directions = ["U", "L", "R"]
    elif action == "R":
        directions = ["R", "U", "D"]
    elif action == "D":
        directions = ["D", "L", "R"]
    else:
        directions = ["L", "U", "D"]

    chosen = random.choices(
        directions,
        weights=[0.8, 0.1, 0.1]
    )[0]

    return move(state, chosen)


def run_episode():

    state = start_state

    steps = 0
    discounted_return = 0
    discount = 1.0

    while steps < 100:

        if state == (0, 3):
            return "goal", steps, discounted_return

        if state == (1, 3):
            return "hole", steps, discounted_return

        action = policy[state]

        next_state = simulate_move(state, action)

        reward = get_reward(next_state)

        discounted_return += discount * reward

        discount *= 0.9

        state = next_state

        steps += 1

    return "time_limit", steps, discounted_return


def run_batches(map_name):

    print("\n====================")
    print(map_name)
    print("====================")

    batch_seeds = [11, 22, 33]

    for batch_seed in batch_seeds:

        goals = 0
        holes = 0
        time_limits = 0

        total_steps = 0
        total_return = 0

        for i in range(50):

            random.seed(1000 * batch_seed + i)

            outcome, steps, discounted_return = run_episode()

            total_steps += steps
            total_return += discounted_return

            if outcome == "goal":
                goals += 1
            elif outcome == "hole":
                holes += 1
            else:
                time_limits += 1

        print(
            f"Batch {batch_seed}: "
            f"Goals={goals}, "
            f"Holes={holes}, "
            f"TimeLimits={time_limits}, "
            f"MeanSteps={round(total_steps/50,2)}, "
            f"MeanReturn={round(total_return/50,2)}"
        )

In [67]:
run_batches("Original Map")
run_batches("Extended Map")


Original Map
Batch 11: Goals=50, Holes=0, TimeLimits=0, MeanSteps=5.0, MeanReturn=1.5
Batch 22: Goals=50, Holes=0, TimeLimits=0, MeanSteps=5.0, MeanReturn=1.5
Batch 33: Goals=50, Holes=0, TimeLimits=0, MeanSteps=5.0, MeanReturn=1.5

Extended Map
Batch 11: Goals=50, Holes=0, TimeLimits=0, MeanSteps=5.0, MeanReturn=1.5
Batch 22: Goals=50, Holes=0, TimeLimits=0, MeanSteps=5.0, MeanReturn=1.5
Batch 33: Goals=50, Holes=0, TimeLimits=0, MeanSteps=5.0, MeanReturn=1.5


In [68]:
import os

os.makedirs("outputs", exist_ok=True)

with open("outputs/value_map_original.txt", "w") as f:
    f.write(
"""4.20 8.00 10.00 0.00
2.78 ##### 5.82 0.00
1.54 2.82 4.24 2.82"""
    )

with open("outputs/policy_map_original.txt", "w", encoding="utf-8") as f:
    f.write(
"""→ → → G
↑ # ↑ H
→ → ↑ ←"""
    )

with open("outputs/value_map_extended.txt", "w") as f:
    f.write(
"""4.20 8.00 10.00 0.00
2.78 ##### 5.82 0.00
1.50 2.82 4.24 2.82"""
    )

with open("outputs/policy_map_extended.txt", "w", encoding="utf-8") as f:
    f.write(
"""→ → → G
↑ # ↑ H
↑ → ↑ ←"""
    )

with open("outputs/rule_checks.txt", "w") as f:
    f.write(
"""Boundary behavior:
move((0,0),'U') -> (0,0)
move((0,0),'L') -> (0,0)

Wall behavior:
move((1,0),'R') -> (1,0)

Ice probabilities:
0.8 + 0.1 + 0.1 = 1.0

Rewards:
Mud = -3
Sand = -2
Water = -5

Terminal handling:
Goal and Hole are terminal states.
No rewards after termination.
"""
    )

print("Part C outputs saved.")

Part C outputs saved.
